# PixiJoy Monocular depth (M2 / H5) — Colab GPU (auto-resume)

**Runtime → GPU → Run all.** Checkpoints live on Google Drive so a disconnect
or re-run **continues from the last saved epoch** (not from scratch).

Stage real data + READY under Drive data/ (see data/README.md).
When finished, download `dist/depth.tflite` only if verify OK → [ml-packs-v1](https://github.com/Saikumar-S0906/Release-/releases/tag/ml-packs-v1).


In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, urllib.request, zipfile, shutil
from pathlib import Path
from google.colab import files

ROOT = Path("/content/pixijoy_depth")
DRIVE = Path("/content/drive/MyDrive/pixijoy_ml/depth")
ROOT.mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "dist").mkdir(parents=True, exist_ok=True)
(DRIVE / "data" / "rgb").mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

ZIP_URL = "https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/depth_lab.zip"
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, "depth_lab.zip")
    with zipfile.ZipFile("depth_lab.zip") as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
    print("extracted lab from Release- main", ok)
except Exception as e:
    print("zip fetch failed:", e)

if not ok:
    print("Upload depth_lab.zip")
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
assert ok, "Could not load lab zip"

# Persist outputs on Drive (survives runtime kill)
if Path("dist").exists() or Path("dist").is_symlink():
    if Path("dist").is_symlink() or Path("dist").is_file():
        Path("dist").unlink()
    else:
        shutil.rmtree("dist")
Path("dist").symlink_to(DRIVE / "dist")
# Prefer Drive data if READY already there
if (DRIVE / "data" / "READY").is_file():
    if Path("data").exists():
        if Path("data").is_symlink() or Path("data").is_file():
            Path("data").unlink()
        else:
            shutil.rmtree("data")
    Path("data").symlink_to(DRIVE / "data")
    print("using Drive data/", DRIVE / "data")
print("Drive ckpt dir", DRIVE / "dist")


In [ ]:
TRAIN_MODE = "quality"  # quality = Release- only; smoke = contract never pin
FORCE_FRESH = False  # True → --no-resume (wipe progress — avoid unless intentional)

from pathlib import Path
from google.colab import files

data = Path("data")
rgb = data / "rgb"
rgb.mkdir(parents=True, exist_ok=True)

if TRAIN_MODE == "quality":
    print("QUALITY: auto-fetch via prepare_quality_data.py (no manual upload)")
    if Path("prepare_quality_data.py").is_file():
        rc = get_ipython().system("python -u prepare_quality_data.py --data ./data")
        if rc != 0:
            raise SystemExit("prepare_quality_data failed")
    elif not (data / "READY").is_file():
        raise SystemExit("No prepare_quality_data.py and no data/READY")
    assert (data / "READY").is_file(), "QUALITY needs data/READY"
    import shutil
    dest = Path("/content/drive/MyDrive/pixijoy_ml/depth/data")
    dest.mkdir(parents=True, exist_ok=True)
    if data.resolve() != dest.resolve():
        for p in data.rglob("*"):
            if p.is_file():
                rel = p.relative_to(data)
                t = dest / rel
                t.parent.mkdir(parents=True, exist_ok=True)
                if not t.is_file():
                    shutil.copy2(p, t)


In [ ]:
OUT = Path("/content/drive/MyDrive/pixijoy_ml/depth/dist/depth.keras")
ckpt = OUT.with_suffix(".ckpt.keras")
state = OUT.parent / "train_state.json"
print("will resume" if ckpt.is_file() else "fresh train", "ckpt=", ckpt, "state=", state.is_file())

flags = f'--out "{OUT}"'
if TRAIN_MODE == "quality":
    flags += " --quality"
else:
    flags += " --smoke"
if FORCE_FRESH:
    flags += " --no-resume"
print("RUN python -u train.py", flags)
rc = get_ipython().system(f"python -u train.py {flags}")
if rc != 0:
    raise SystemExit("train failed — fix data, re-run this cell (Drive auto-resume)")
rc = get_ipython().system(f'python -u export_tflite.py --keras "{OUT}" --out "dist/depth.tflite"')
if rc != 0:
    raise SystemExit("export failed — refusing untrained weights is correct; re-train")
vflags = "--quality" if TRAIN_MODE == "quality" else ""
rc = get_ipython().system(f"python -u verify_contract.py {vflags}")
if rc != 0:
    raise SystemExit("verify failed — do NOT pin to Release-")
from pathlib import Path
import hashlib
p = Path("dist/depth.tflite")
if not p.is_file():
    raise SystemExit("missing dist/depth.tflite")
print("READY", p, p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download("dist/depth.tflite")
print("Upload to ml-packs-v1 as depth.tflite ONLY if verify printed OK")
